# Anonimize fMRI (HCL) 

In [16]:
import matplotlib
#enable detached figure
matplotlib.use('TkAgg')

from module.utils import *
import glob

#set path
path = "/home/caredda/Videos/DATA_TO_UPLOAD/"
#patient ID
array_Patient_id = np.array(["P35", "P51", "P59", "P60", "P61", "P64", "P75", "P77", "P78", "P83"])

for Patient_id in array_Patient_id:
    # Get fMRI filenames
    filename_fMRI = glob.glob(path+Patient_id+"/stats/*.nii")


    #text position in MRI slices
    l1 = 118
    l2 = 592
    c1 = 11
    c2 = 389

    #Create output directory
    output_dir = path+Patient_id+"/stats/anonym/"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    #Loop over fMRI files
    for MRI_filename in filename_fMRI:
        print(MRI_filename)

        #Load fMRI data
        nib_img = nib.load(MRI_filename)
        vol = np.asanyarray(nib_img.dataobj)
        rgb_array = np.stack([vol['R'], vol['G'], vol['B']], axis=-1)
        


        plt.close('all')
        # data = rgb_array[:,:,-1,:]
        # plt.figure()
        # plt.imshow(data, cmap='gray')
        # plt.axis('off')
        # plt.title("fMRI data")
        # plt.show(block=True)


        #Init output volumes
        fMRI_mask = np.zeros(rgb_array.shape[:3], dtype=np.uint8)
        anatomical_vol = np.zeros(rgb_array.shape[:3], dtype=np.uint8)

        #Loop over slices
        for i in range(rgb_array.shape[2]):

            data = rgb_array[:,:,i,:]

            # Mask text in slices
            data[:,0:c1] = 0
            data[:,c2:] = 0
            data[0:l1,:] = 0
            data[l2:,:] = 0

            #Get fMRI mask (red-to-yellow pixels)
            fMRI_mask[:,:,i] = (data[:,:,0] != data[:,:,1]) | (data[:,:,1] != data[:,:,2]) | (data[:,:,2] != data[:,:,0])
            anatomical_vol[:,:,i] = data[:,:,0]


            # plt.figure()
            # plt.subplot(131)
            # plt.imshow(data, cmap='gray')
            # plt.axis('off')
            # plt.title("fMRI data")
            # plt.subplot(132)
            # plt.imshow(fMRI_mask[:,:,i], cmap='gray')
            # plt.axis('off')
            # plt.title("fMRI mask")
            # plt.subplot(133)
            # plt.imshow(anatomical_vol[:,:,i], cmap='gray')
            # plt.axis('off')
            # plt.title("Anatomical mask")
            # plt.show()

        fMRI_mask = fMRI_mask.astype(np.uint8)

        # Create new nii volumes with original affine and header
        nib_anatomical_vol = nib.Nifti1Image(anatomical_vol, nib_img.affine, nib_img.header)
        nib_fMRI_mask = nib.Nifti1Image(fMRI_mask, nib_img.affine, nib_img.header)

        nib_anatomical_vol.set_data_dtype(np.uint8)
        nib_fMRI_mask.set_data_dtype(np.uint8)


        # Save to new file
        nib.save(nib_anatomical_vol, output_dir + os.path.basename(MRI_filename).split('.nii')[0] + '_anatomical.nii')
        nib.save(nib_fMRI_mask, output_dir + os.path.basename(MRI_filename).split('.nii')[0] + '_fMRI_stats.nii')



/home/caredda/Videos/DATA_TO_UPLOAD/P35/stats/40100_IRMf_motrice_main_gauche.nii
/home/caredda/Videos/DATA_TO_UPLOAD/P35/stats/40102_IRMf_langage_comprehension_de_phrases.nii
/home/caredda/Videos/DATA_TO_UPLOAD/P35/stats/40101_IRMf_langage_generation_de_verbes.nii
/home/caredda/Videos/DATA_TO_UPLOAD/P35/stats/40103_IRMf_langage_generation_de_phrases.nii
/home/caredda/Videos/DATA_TO_UPLOAD/P51/stats/20100_IRMf_motrice_main_G.nii
/home/caredda/Videos/DATA_TO_UPLOAD/P51/stats/20102_IRMf_langage_generation_de_verbes.nii
/home/caredda/Videos/DATA_TO_UPLOAD/P51/stats/20103_IRMf_langage_comprehension_de_phrases.nii
/home/caredda/Videos/DATA_TO_UPLOAD/P51/stats/20101_IRMf_motrice_joue_G.nii
/home/caredda/Videos/DATA_TO_UPLOAD/P51/stats/20104_IRMf_langage_generation_de_phrases.nii
/home/caredda/Videos/DATA_TO_UPLOAD/P59/stats/20103_IRMf_motrice_main_GAUCHE.nii
/home/caredda/Videos/DATA_TO_UPLOAD/P59/stats/20100_IRMf_langage_generation_de_verbes.nii
/home/caredda/Videos/DATA_TO_UPLOAD/P59/stats/

# Find the slice in MRI that matches anatomical MRI

In [61]:
import matplotlib
#enable detached figure
matplotlib.use('TkAgg')

from module.utils import *
import glob
import cv2
from skimage.metrics import normalized_mutual_information, normalized_root_mse

def normalized_cross_correlation(img1, img2):
    img1 = img1.astype(np.float32)
    img2 = img2.astype(np.float32)

    mean1 = img1.mean()
    mean2 = img2.mean()
    std1 = img1.std()
    std2 = img2.std()

    ncc = np.mean((img1 - mean1) * (img2 - mean2)) / (std1 * std2 + 1e-8)
    return ncc

def create_itk_transverse_slice(vol_itk, id_slice, h_flip = False):
    np_vol = itk.GetArrayFromImage(vol_itk)

    #Create transverse slice
    np_transverse_slice = np_vol[id_slice,:,:]
    if h_flip:
        np_transverse_slice = np.flip(np_transverse_slice, axis=1)
    transverse_slice  = itk.GetImageFromArray(np.ascontiguousarray(np_transverse_slice))
    transverse_slice .SetSpacing([vol_itk.GetSpacing()[1], vol_itk.GetSpacing()[2]])

    return transverse_slice

def create_np_transverse_slice(vol_itk, id_slice, h_flip = False):
    np_vol = itk.GetArrayFromImage(vol_itk)

    #Create transverse slice
    np_transverse_slice = np_vol[id_slice,:,:]
    if h_flip:
        np_transverse_slice = np.flip(np_transverse_slice, axis=1)
    

    return np_transverse_slice


def rigid_register_2d(fixed, moving, metric_func, log = False):

    # Get mask images
    mask_fixed = fixed > 0
    mask_moving = moving > 0

    mask_fixed = mask_fixed.astype(np.uint8)*255
    mask_moving = mask_moving.astype(np.uint8)*255

    if mask_fixed.sum() == 0 or mask_moving.sum() == 0:
        return moving, 0

    # Find contours
    contours_fixed, _ = cv2.findContours(mask_fixed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours_moving, _ = cv2.findContours(mask_moving, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Get the bounding box of the largest contour
    largest_contour_fixed = max(contours_fixed, key=cv2.contourArea)
    largest_contour_moving = max(contours_moving, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour_fixed)
    x_m, y_m, w_m, h_m = cv2.boundingRect(largest_contour_moving)

    if (w<10) or (h<10):
        return moving, 0
    
    # Crop the images to the bounding box
    moving_cropped = moving[y_m:y_m+h_m, x_m:x_m+w_m]

    if log:
        print("moving", w_m, h_m )
        print("fixed", w, h )


    # Resize the moving image to the size of the fixed image
    moving_resized = cv2.resize(moving_cropped, (w, h), interpolation=cv2.INTER_NEAREST)

    #define registered image
    registered = np.zeros_like(fixed)
    registered[y:y+h, x:x+w] = moving_resized

    #normalize images
    norm_fixed = 255*(fixed - fixed.min()) / (fixed.max() - fixed.min())
    norm_moving = 255*(registered - registered.min()) / (registered.max() - registered.min())

    size_k = 13
    norm_fixed = cv2.GaussianBlur(norm_fixed.astype(np.uint8), (size_k, size_k), 0)

    # norm_moving = cv2.GaussianBlur(norm_fixed.astype(np.uint8), (5, 5), 0)


    # Calculate the metric
    metric = metric_func(norm_fixed, norm_moving)
    return norm_moving, metric




#set path
path = "/home/caredda/Videos/DATA_TO_UPLOAD/"

#patient ID
# array_Patient_id = np.array(["P35", "P51", "P59", "P60", "P61", "P64", "P75", "P77", "P78", "P83"])
Patient_id = "P35"

# Get fMRI filenames
filename_fMRI = glob.glob(path+Patient_id+"/stats/anonym/*_anatomical.nii")

MRI_filename = filename_fMRI[0]


# Load anatomical MRI volumes
vol_itk_fMRI = itk.imread(MRI_filename, itk.F)
vol_itk_fMRI_stats = itk.imread(MRI_filename.split("_anatomical.nii")[0]+"_fMRI_stats.nii", itk.F)
vol_itk_FLAIR = itk.imread(path+Patient_id+"/stats/FLAIR.nii", itk.F)



# for id_slice_fMRI in range(vol_itk_fMRI.shape[0]):
#     #Create transverse slice
#     transverse_slice_fMRI  = create_np_transverse_slice(vol_itk_fMRI, id_slice_fMRI, h_flip = True)
#     transverse_slice_fMRI_stats  = create_np_transverse_slice(vol_itk_fMRI_stats, id_slice_fMRI, h_flip = True)

#     print("fMRI slice", id_slice_fMRI,"nb of voxels", transverse_slice_fMRI_stats.sum())





#Create transverse slice
id_slice_fMRI = 27
transverse_slice_fMRI  = create_np_transverse_slice(vol_itk_fMRI, id_slice_fMRI, h_flip = True)
transverse_slice_fMRI_stats  = create_np_transverse_slice(vol_itk_fMRI_stats, id_slice_fMRI, h_flip = True)


metric_func = normalized_cross_correlation
metric = np.array([])

#Loop over slices
for slice_id_FLAIR in range(0,250):#range(vol_itk_FLAIR.shape[0]):
    #Create transverse slice
    transverse_slice_FLAIR  = create_np_transverse_slice(vol_itk_FLAIR, slice_id_FLAIR)

    #Process registration
    img_registered, metric_val =  rigid_register_2d(transverse_slice_FLAIR, transverse_slice_fMRI, metric_func)
    metric = np.append(metric, metric_val) 

#Apply registration to the correct slice
transverse_slice_FLAIR  = create_np_transverse_slice(vol_itk_FLAIR, metric.argmax())
img_registered, metric_val =  rigid_register_2d(transverse_slice_FLAIR, transverse_slice_fMRI, metric_func, log=True)

print(metric.argmax())


plt.close('all')
plt.figure()


plt.subplot(221)
plt.imshow(transverse_slice_FLAIR, cmap='gray')
plt.axis('off')
plt.title("Transverse slice from FLAIR")

plt.subplot(222)
plt.imshow(transverse_slice_fMRI, cmap='gray')
plt.axis('off')
plt.title("Transverse slice from anatomical used in fMRI")

plt.subplot(223)
plt.imshow(img_registered, cmap='gray')
plt.axis('off')
plt.title("Registered fMRI slice")

plt.subplot(224)
plt.plot(metric)
plt.title("Registration metric")
plt.xlabel("Slice ID")
plt.ylabel("Metric value")
plt.show(block=True)

moving 180 230
fixed 236 195
244


In [60]:
#Apply registration to the correct slice
transverse_slice_FLAIR  = create_np_transverse_slice(vol_itk_FLAIR, 247)
img_registered, metric_val =  rigid_register_2d(transverse_slice_FLAIR, transverse_slice_fMRI, metric_func)

print("metric ncc", metric_val)

plt.close('all')
plt.figure()


plt.subplot(121)
plt.imshow(transverse_slice_FLAIR, cmap='gray')
plt.axis('off')
plt.title("Transverse slice from FLAIR")

plt.subplot(122)
plt.imshow(img_registered, cmap='gray')
plt.axis('off')
plt.title("Transverse slice from anatomical used in fMRI")

plt.show(block=True)

metric ncc 0.43538736383229704
